# Exploración de Datos y Resultados (DL-2026-02-L02)

Este notebook carga tanto el dataset original (`.sav`) como las tablas de resultados generadas en `results/`.

In [1]:
from pathlib import Path
import pandas as pd
import pyreadstat
from IPython.display import display

# Detectar la raíz del repositorio automáticamente (funciona desde dataset/ o desde la raíz)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "dataset" else Path.cwd()
print(f"Raíz del repositorio: {REPO_ROOT}")

Raíz del repositorio: /var/home/Helo/Documents/DL-2026-02-L02


## 1. Exploración del Dataset (`15 atributos R0-R5.sav`)

In [2]:
DATA_PATH = REPO_ROOT / "dataset" / "15 atributos R0-R5.sav"
df_datos, meta = pyreadstat.read_sav(str(DATA_PATH))

print(f"Dimensiones del dataset: {df_datos.shape[0]} filas x {df_datos.shape[1]} columnas")
display(df_datos.head(10))

# Distribución de clases en los 6 objetivos
targets = ["GDS", "GDS_R1", "GDS_R2", "GDS_R3", "GDS_R4", "GDS_R5"]
dist_objetivos = (
    pd.concat({t: df_datos[t].value_counts().sort_index() for t in targets}, axis=1)
    .fillna(0)
    .astype(int)
)
print("\n=== Conteo de muestras por clase en cada objetivo ===")
display(dist_objetivos)

Dimensiones del dataset: 1119 filas x 22 columnas


,ID,Día,Mes,Año,Estación,País,Ciudad,CalleLugar,NumeroPiso,Miguel2,...,Imperial2,A682,Caldera2,Copiapo2,GDS,GDS_R1,GDS_R2,GDS_R3,GDS_R4,GDS_R5
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,2.0,2.0
1,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
2,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,4.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,0.0,1.0,1.0,0.0,2.0,1.0,1.0,1.0,2.0,2.0
4,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,2.0,1.0,1.0,1.0,2.0,2.0
5,6.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,3.0,1.0,2.0,1.0,2.0,2.0
6,7.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,2.0
7,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,9.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,10.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0



=== Conteo de muestras por clase en cada objetivo ===


,GDS,GDS_R1,GDS_R2,GDS_R3,GDS_R4,GDS_R5
1.0,149,947,649,947,149,149
2.0,500,150,298,0,906,798
3.0,298,22,172,172,64,172
4.0,108,0,0,0,0,0
5.0,42,0,0,0,0,0
6.0,20,0,0,0,0,0
7.0,2,0,0,0,0,0


## 2. Tablas de Resultados de los Experimentos (`results/`)

In [3]:
RESULTS_DIRS = [
    REPO_ROOT / "results",
    REPO_ROOT / "results" / "coral_pesos",  # Se incluye automáticamente si existe
]

METRIC_LABELS = {
    "accuracy": "Acc",
    "f1_macro": "F1_m",
    "balanced_accuracy": "BalAcc",
    "mae_ordinal": "MAE",
    "qwk": "QWK",
    "accuracy_pm1": "Acc±1",
    "errores_graves": "Err≥2",
}


def cargar_csvs(nombre_archivo: str, directorios: list[Path]) -> pd.DataFrame:
    """Concatena un mismo archivo CSV desde uno o más directorios de resultados."""
    tablas = []
    for d in directorios:
        ruta = d / nombre_archivo
        if ruta.exists():
            df = pd.read_csv(ruta)
            df["source_dir"] = str(d.relative_to(REPO_ROOT))
            tablas.append(df)
    if not tablas:
        raise FileNotFoundError(f"No se encontró {nombre_archivo} en {directorios}")
    return pd.concat(tablas, ignore_index=True)


def nombre_modelo(row: pd.Series) -> str:
    """Etiqueta legible del modelo según algoritmo y uso de pesos."""
    if row["algorithm"] == "conv1d_softmax":
        return "Conv1D + Softmax"
    if row.get("use_class_weights", False):
        return "Conv1D + CORAL + pesos"
    return "Conv1D + CORAL"


def formatear_tabla_resumen(df_res: pd.DataFrame) -> pd.DataFrame:
    """Construye la tabla comparativa (media ± std) por objetivo y modelo."""
    df = df_res.copy()
    df["Modelo"] = df.apply(nombre_modelo, axis=1)

    filas = []
    for _, row in df.iterrows():
        item = {
            "Objetivo": row["target"],
            "Modelo": row["Modelo"],
            "Config más elegida": (
                f"conv={row['conv_channels']}, k={row['kernel_size']}, "
                f"h={row['hidden_dim']}, drop={row['dropout']}, lr={row['learning_rate']}"
            ),
        }
        for key, label in METRIC_LABELS.items():
            mean = row[key]
            std = row[f"{key}_std"]
            item[label] = f"{mean:.4f} ± {std:.4f}"
        filas.append(item)

    return (
        pd.DataFrame(filas)
        .drop_duplicates(subset=["Objetivo", "Modelo"], keep="last")
        .set_index(["Objetivo", "Modelo"])
    )


# 1. Tablas crudas (valores numéricos completos)
df_resultados = cargar_csvs("resultados.csv", RESULTS_DIRS)
df_seleccion_hp = cargar_csvs("seleccion_hp.csv", RESULTS_DIRS)
df_ranking_f1 = cargar_csvs("ranking_f1.csv", RESULTS_DIRS)
df_ranking_mae = cargar_csvs("ranking_mae.csv", RESULTS_DIRS)

# 2. Tabla formateada para el informe (media ± std)
tabla_informe = formatear_tabla_resumen(df_resultados)

print("=== Tabla Resumen para el Informe (media ± std) ===")
display(tabla_informe)

print("\n=== Resultados Numéricos Crudos ===")
display(df_resultados)

print("\n=== Selección de Hiperparámetros por Fold ===")
display(df_seleccion_hp)

print("\n=== Ranking por F1 Macro (mayor es mejor) ===")
display(df_ranking_f1)

print("\n=== Ranking por MAE Ordinal (menor es mejor) ===")
display(df_ranking_mae)

=== Tabla Resumen para el Informe (media ± std) ===


,,Config más elegida,Acc,F1_m,BalAcc,MAE,QWK,Acc±1,Err≥2
Objetivo,Modelo,,,,,,,,
GDS,Conv1D + Softmax,"conv=16-32, k=3, h=32, drop=0.15, lr=0.001",0.5683 ± 0.0209,0.3347 ± 0.0001,0.3458 ± 0.0122,0.4737 ± 0.0147,0.7051 ± 0.0112,0.9625 ± 0.0071,0.0375 ± 0.0071
GDS_R1,Conv1D + Softmax,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.8972 ± 0.0225,0.7493 ± 0.0295,0.7331 ± 0.0386,0.1037 ± 0.0224,0.6912 ± 0.0573,0.9991 ± 0.0018,0.0009 ± 0.0018
GDS_R2,Conv1D + Softmax,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.7247 ± 0.0354,0.6608 ± 0.0355,0.6428 ± 0.0379,0.2914 ± 0.0409,0.6915 ± 0.0572,0.9839 ± 0.0073,0.0161 ± 0.0073
GDS_R3,Conv1D + Softmax,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.9035 ± 0.0210,0.7851 ± 0.0442,0.7479 ± 0.0380,0.0965 ± 0.0210,0.5727 ± 0.0880,1.0000 ± 0.0000,0.0000 ± 0.0000
GDS_R4,Conv1D + Softmax,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.8302 ± 0.0094,0.5139 ± 0.0312,0.5162 ± 0.0381,0.1698 ± 0.0094,0.2930 ± 0.0508,1.0000 ± 0.0000,0.0000 ± 0.0000
GDS_R5,Conv1D + Softmax,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.7748 ± 0.0066,0.5037 ± 0.0143,0.5095 ± 0.0178,0.2252 ± 0.0066,0.4274 ± 0.0320,1.0000 ± 0.0000,0.0000 ± 0.0000
GDS,Conv1D + CORAL,"conv=16-32, k=3, h=32, drop=0.15, lr=0.001",0.4120 ± 0.0112,0.2955 ± 0.0157,0.3684 ± 0.0599,0.6890 ± 0.0301,0.6853 ± 0.0134,0.9088 ± 0.0144,0.0912 ± 0.0144
GDS_R1,Conv1D + CORAL,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.8883 ± 0.0150,0.6792 ± 0.0283,0.6810 ± 0.0477,0.1144 ± 0.0156,0.6307 ± 0.0676,0.9973 ± 0.0022,0.0027 ± 0.0022
GDS_R2,Conv1D + CORAL,"conv=8-16, k=3, h=32, drop=0.15, lr=0.001",0.7319 ± 0.0232,0.6494 ± 0.0280,0.6428 ± 0.0342,0.2949 ± 0.0276,0.6831 ± 0.0440,0.9732 ± 0.0064,0.0268 ± 0.0064



=== Resultados Numéricos Crudos ===


,algorithm,target,hp_mode,conv_channels,kernel_size,hidden_dim,dropout,learning_rate,weight_decay,use_class_weights,...,accuracy_std,balanced_accuracy_std,precision_macro_std,recall_macro_std,f1_macro_std,mae_ordinal_std,qwk_std,accuracy_pm1_std,errores_graves_std,source_dir
0,conv1d_softmax,GDS,grid,16-32,3,32,0.15,0.001,0.0001,False,...,0.020940,0.012174,0.010005,0.012174,0.000085,0.014722,0.011162,0.007116,0.007116,results
1,conv1d_softmax,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022541,0.038598,0.071107,0.038598,0.029519,0.022431,0.057268,0.001786,0.001786,results
2,conv1d_softmax,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.035420,0.037873,0.030927,0.037873,0.035538,0.040869,0.057227,0.007258,0.007258,results
3,conv1d_softmax,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.020987,0.038050,0.062474,0.038050,0.044162,0.020987,0.088038,0.000000,0.000000,results
4,conv1d_softmax,GDS_R4,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.009369,0.038111,0.021406,0.038111,0.031183,0.009369,0.050831,0.000000,0.000000,results
5,conv1d_softmax,GDS_R5,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.006640,0.017784,0.009858,0.017784,0.014253,0.006640,0.032018,0.000000,0.000000,results
6,conv1d_coral,GDS,grid,16-32,3,32,0.15,0.001,0.0001,False,...,0.011249,0.059852,0.032677,0.059852,0.015720,0.030106,0.013415,0.014380,0.014380,results
7,conv1d_coral,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.014973,0.047689,0.088302,0.047689,0.028256,0.015632,0.067587,0.002187,0.002187,results
8,conv1d_coral,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.023193,0.034211,0.031193,0.034211,0.028022,0.027585,0.044037,0.006359,0.006359,results
9,conv1d_coral,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022078,0.040808,0.056408,0.040808,0.046587,0.022078,0.092784,0.000000,0.000000,results



=== Selección de Hiperparámetros por Fold ===


,algorithm,target,outer_fold,inner_skipped,conv_channels,kernel_size,hidden_dim,dropout,learning_rate,weight_decay,beta,inner_mae,inner_qwk,test_f1_macro,test_mae_ordinal,source_dir
0,conv1d_softmax,GDS,1,True,16-32,3,32,0.15,0.001,0.0001,0.99,NaN,NaN,0.334795,0.458929,results
1,conv1d_softmax,GDS,2,True,16-32,3,32,0.15,0.001,0.0001,0.99,NaN,NaN,0.334626,0.488372,results
2,conv1d_softmax,GDS_R1,1,False,16-32,3,32,0.15,0.001,0.0001,0.99,0.110611,0.666827,0.769117,0.075893,results
3,conv1d_softmax,GDS_R1,2,False,8-16,3,32,0.15,0.001,0.0001,0.99,0.106141,0.684708,0.739542,0.102679,results
4,conv1d_softmax,GDS_R1,3,False,8-16,3,32,0.15,0.001,0.0001,0.99,0.100559,0.681491,0.711618,0.107143,results
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,conv1d_coral,GDS_R5,1,False,16-32,3,64,0.30,0.001,0.0001,0.99,0.500554,0.455477,0.600199,0.379464,results/coral_pesos
77,conv1d_coral,GDS_R5,2,False,16-32,3,64,0.30,0.001,0.0001,0.99,0.491609,0.431168,0.543654,0.455357,results/coral_pesos
78,conv1d_coral,GDS_R5,3,False,16-32,3,64,0.30,0.001,0.0001,0.99,0.505020,0.459893,0.488487,0.495536,results/coral_pesos
79,conv1d_coral,GDS_R5,4,False,16-32,3,32,0.15,0.001,0.0001,0.99,0.484897,0.452809,0.541369,0.441964,results/coral_pesos



=== Ranking por F1 Macro (mayor es mejor) ===


,algorithm,target,hp_mode,conv_channels,kernel_size,hidden_dim,dropout,learning_rate,weight_decay,use_class_weights,...,accuracy_std,balanced_accuracy_std,precision_macro_std,recall_macro_std,f1_macro_std,mae_ordinal_std,qwk_std,accuracy_pm1_std,errores_graves_std,source_dir
0,conv1d_coral,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022078,0.040808,0.056408,0.040808,0.046587,0.022078,0.092784,0.000000,0.000000,results
1,conv1d_softmax,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.020987,0.038050,0.062474,0.038050,0.044162,0.020987,0.088038,0.000000,0.000000,results
2,conv1d_softmax,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022541,0.038598,0.071107,0.038598,0.029519,0.022431,0.057268,0.001786,0.001786,results
3,conv1d_coral,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.014973,0.047689,0.088302,0.047689,0.028256,0.015632,0.067587,0.002187,0.002187,results
4,conv1d_softmax,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.035420,0.037873,0.030927,0.037873,0.035538,0.040869,0.057227,0.007258,0.007258,results
5,conv1d_coral,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.023193,0.034211,0.031193,0.034211,0.028022,0.027585,0.044037,0.006359,0.006359,results
6,conv1d_coral,GDS_R4,grid,16-32,5,32,0.15,0.001,0.0001,False,...,0.026688,0.032247,0.022080,0.032247,0.024286,0.026688,0.029073,0.000000,0.000000,results
7,conv1d_coral,GDS_R5,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.043037,0.037127,0.036310,0.037127,0.032013,0.046019,0.040867,0.005653,0.005653,results
8,conv1d_softmax,GDS_R4,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.009369,0.038111,0.021406,0.038111,0.031183,0.009369,0.050831,0.000000,0.000000,results
9,conv1d_softmax,GDS_R5,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.006640,0.017784,0.009858,0.017784,0.014253,0.006640,0.032018,0.000000,0.000000,results



=== Ranking por MAE Ordinal (menor es mejor) ===


,algorithm,target,hp_mode,conv_channels,kernel_size,hidden_dim,dropout,learning_rate,weight_decay,use_class_weights,...,accuracy_std,balanced_accuracy_std,precision_macro_std,recall_macro_std,f1_macro_std,mae_ordinal_std,qwk_std,accuracy_pm1_std,errores_graves_std,source_dir
0,conv1d_coral,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022078,0.040808,0.056408,0.040808,0.046587,0.022078,0.092784,0.000000,0.000000,results
1,conv1d_softmax,GDS_R3,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.020987,0.038050,0.062474,0.038050,0.044162,0.020987,0.088038,0.000000,0.000000,results
2,conv1d_softmax,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.022541,0.038598,0.071107,0.038598,0.029519,0.022431,0.057268,0.001786,0.001786,results
3,conv1d_coral,GDS_R1,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.014973,0.047689,0.088302,0.047689,0.028256,0.015632,0.067587,0.002187,0.002187,results
4,conv1d_softmax,GDS_R4,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.009369,0.038111,0.021406,0.038111,0.031183,0.009369,0.050831,0.000000,0.000000,results
5,conv1d_softmax,GDS_R5,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.006640,0.017784,0.009858,0.017784,0.014253,0.006640,0.032018,0.000000,0.000000,results
6,conv1d_softmax,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.035420,0.037873,0.030927,0.037873,0.035538,0.040869,0.057227,0.007258,0.007258,results
7,conv1d_coral,GDS_R2,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.023193,0.034211,0.031193,0.034211,0.028022,0.027585,0.044037,0.006359,0.006359,results
8,conv1d_coral,GDS_R4,grid,16-32,5,32,0.15,0.001,0.0001,False,...,0.026688,0.032247,0.022080,0.032247,0.024286,0.026688,0.029073,0.000000,0.000000,results
9,conv1d_coral,GDS_R5,grid,8-16,3,32,0.15,0.001,0.0001,False,...,0.043037,0.037127,0.036310,0.037127,0.032013,0.046019,0.040867,0.005653,0.005653,results
